# 04 — Preprocessing & Feature Engineering

**Owner:** Umer's lane — imputation, scaling, encoding, feature engineering, `src/pipeline.py`.

**Input (handoff from notebook 03):** `data/processed/eligible_features.json` (the 14 raw leakage-safe features) and the split decision in `split_cycles.json`.

**Output:** the two engineered current features, the final 16-feature list (saved back to `eligible_features.json`), the fixed train/test split (`train.parquet`, `test.parquet`), and the preprocessing pipeline builder in `src/pipeline.py`.

**Viva questions to be ready for:** Why fit only on train? Why median imputation / RobustScaler? Which engineered features and why?

Every important choice has a decision-log cell (what we decided, the evidence, the alternative considered, why it was rejected).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedGroupKFold

from src.config import RANDOM_STATE, PROCESSED_DATA_DIR
from src.features import add_engineered_features
from src.pipeline import load_data, build_preprocessing_pipeline

pd.set_option("display.max_columns", None)

In [3]:
df = load_data()
df.shape

(7409, 24)

## Load the eligible-feature list (from notebook 03)

The 14 raw features that survived the leakage study.

In [4]:
ENGINEERED_FEATURES = ["total_joint_current", "total_joint_current_delta3"]

with open(PROCESSED_DATA_DIR / "eligible_features.json") as f:
    # Keep only the 14 raw features: this notebook adds the engineered ones below and re-saves the final list
    eligible_features = [c for c in json.load(f) if c not in ENGINEERED_FEATURES]

eligible_features

['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'grip_lost']

## Drop the unrecoverable rows

Per notebook 01, the 54 rows with missing values are also missing the target (46 are missing every sensor), so they cannot be used for supervised training whatever the imputation strategy. They are dropped here (7,409 → 7,355 rows).

In [5]:
df = df.dropna(subset=["Robot_ProtectiveStop"]).reset_index(drop=True)
df.shape

(7355, 24)

## Feature engineering

`total_joint_current` (sum of |Current_J0..J5|): a non-leaky summary of how hard the joints are working. It is higher at stop rows than at normal rows (median 5.99 vs 4.31, notebook 03), unlike speed, which is a consequence of the stop.

`total_joint_current_delta3`: the change in `total_joint_current` over the last 3 readings within the same cycle, i.e. whether the total current is rising or falling. *Revised after notebooks 03 and 07:* this feature was introduced to capture "current trending up before a stop", but the same-cycle-position control in notebook 03 shows that the rise before a stop is largely the normal start-of-cycle current wave, so the feature is better described as a **trend / cycle-phase signal**. It still earns its place: without it, the cross-validated PR-AUC of the tuned XGBoost falls from 0.473 to 0.414 (notebook 07, section 2). It is not evidence of an early-warning precursor.

`total_joint_current_delta3` introduces genuine `NaN`s for the first 3 rows of every cycle (no earlier reading within that cycle to diff against), unlike the raw eligible features, which have no missing values. That is why the median imputer below does real work for this column and is not only a defensive step.

The features are built by `src/features.py`, the same code the prediction API uses, so training and serving cannot drift apart. This must run before the split/imputation/scaling cells below, since those fit on whatever `eligible_features` contains at that point. The final list is saved back to `eligible_features.json` so notebooks 05-07 load the same 16 features.

In [6]:
# Engineered features come from src/features.py (also used by the API at serving time)
df = add_engineered_features(df, group_col="cycle")

# Final 16-feature list, saved for notebooks 05-07
eligible_features = eligible_features + ENGINEERED_FEATURES
with open(PROCESSED_DATA_DIR / "eligible_features.json", "w") as f:
    json.dump(eligible_features, f, indent=2)

# NaN for the first 3 rows of each cycle (no prior reading within that cycle to diff against)
print("NaNs introduced by the lag feature:", df["total_joint_current_delta3"].isna().sum())
eligible_features

NaNs introduced by the lag feature: 720


['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'grip_lost',
 'total_joint_current',
 'total_joint_current_delta3']

## Apply the split decision (from notebook 03)

Reproduce the exact split from notebook 03 (the last 20% of cycles by start time are the test set), so every notebook works on identical rows. Everything below is fit on **train only**.

In [7]:
with open(PROCESSED_DATA_DIR / "split_cycles.json") as f:
    split_info = json.load(f)

train_cycles = split_info["train_cycles"]
test_cycles = split_info["test_cycles"]

train_df = df[df["cycle"].isin(train_cycles)].reset_index(drop=True)
test_df = df[df["cycle"].isin(test_cycles)].reset_index(drop=True)

print("train rows:", len(train_df), "| test rows:", len(test_df))

train rows: 5468 | test rows: 1887


## Imputation

Median imputation for the sensor columns (robust to outliers, unlike the mean), fit on train only and then applied to both train and test. Only `total_joint_current_delta3` has missing values (the first 3 rows of each cycle).

In [8]:
imputer = SimpleImputer(strategy="median")
imputer.fit(train_df[eligible_features])

SimpleImputer(strategy='median')

## Scaling

`RobustScaler` (median and IQR, so the outliers found in notebook 02 do not dominate the scale), fit on train only. `grip_lost` is excluded because it is a binary flag.

In [9]:
continuous_cols = [c for c in eligible_features if c != "grip_lost"]
binary_cols = ["grip_lost"]

scaler = RobustScaler()
scaler.fit(train_df[continuous_cols])

RobustScaler()

## Feature selection: correlation check

Is any feature redundant? The first table shows how the engineered feature relates to the other continuous features; the second lists the most correlated pairs among all 15 continuous features and summarises the temperature block. Any drops are decided in the decision log below.

In [10]:
correlation = train_df[continuous_cols].corr()
print("Correlation with total_joint_current:")
print(correlation["total_joint_current"].sort_values(ascending=False).round(3).to_string())

# All pairs, strongest first
pairs = correlation.where(np.triu(np.ones(correlation.shape, dtype=bool), k=1)).stack().rename("r").to_frame()
pairs["abs_r"] = pairs["r"].abs()
print("\nMost correlated pairs among the continuous features:")
print(pairs.sort_values("abs_r", ascending=False).head(8).round(3).to_string())

temperature_cols = [c for c in continuous_cols if c.startswith("Temperature_")]
in_temp = pairs.index.get_level_values(0).isin(temperature_cols) | pairs.index.get_level_values(1).isin(temperature_cols)
both_temp = pairs.index.get_level_values(0).isin(temperature_cols) & pairs.index.get_level_values(1).isin(temperature_cols)
print(f"\nTemperature pairs: {both_temp.sum()}, all with |r| >= {pairs.loc[both_temp, 'abs_r'].min():.3f}")
print(f"Largest |r| between a temperature and a non-temperature feature: {pairs.loc[in_temp & ~both_temp, 'abs_r'].max():.3f}")
print("Strongest pairs not involving a temperature:")
print(pairs.loc[~in_temp].sort_values("abs_r", ascending=False).head(4).round(3).to_string())

Correlation with total_joint_current:
total_joint_current           1.000
total_joint_current_delta3    0.721
Current_J0                    0.039
Current_J5                    0.019
Current_J4                    0.018
Temperature_J4                0.004
Temperature_J5                0.003
Temperature_J3                0.003
Temperature_J2                0.001
Temperature_J1               -0.000
Temperature_T0               -0.000
Tool_current                 -0.055
Current_J3                   -0.351
Current_J2                   -0.590
Current_J1                   -0.652

Most correlated pairs among the continuous features:
                                   r  abs_r
Temperature_J3 Temperature_J4  1.000  1.000
Temperature_J1 Temperature_J2  1.000  1.000
Temperature_T0 Temperature_J1  0.999  0.999
Temperature_J2 Temperature_J3  0.999  0.999
Temperature_J4 Temperature_J5  0.999  0.999
Temperature_J3 Temperature_J5  0.999  0.999
Temperature_T0 Temperature_J2  0.999  0.999
Temperature_J2 T

## Assemble the preprocessing pipeline

Imputer + scaler + passthrough for `grip_lost`, combined in one `ColumnTransformer` (`build_preprocessing_pipeline()` in `src/pipeline.py`) so notebooks 05-07 and the backend all reuse the exact same object. Per the working agreement, shared code lives in `src/` and is never copy-pasted between notebooks.

In [11]:
preprocessing_pipeline = build_preprocessing_pipeline(continuous_cols, binary_cols)
preprocessing_pipeline.fit(train_df[eligible_features])

output_columns = continuous_cols + binary_cols
output_columns

['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'total_joint_current',
 'total_joint_current_delta3',
 'grip_lost']

## Save the fixed train/test split

Persist the split so every model notebook (05-07) loads identical rows and never re-splits. The `cycle` column is kept alongside the features for the group-aware cross-validation in notebook 05.

In [12]:
train_df.to_parquet(PROCESSED_DATA_DIR / "train.parquet", index=False)
test_df.to_parquet(PROCESSED_DATA_DIR / "test.parquet", index=False)
print("saved: data/processed/train.parquet | data/processed/test.parquet")

saved: data/processed/train.parquet | data/processed/test.parquet


## Decision log

*Entries are Umer's. Two were reworded by Bandara on 25 Sep after notebooks 03 and 07 changed the evidence (marked "reworded"); conclusions that still hold are kept.*

### Decision: Median imputation for continuous features, passthrough (no scaling) for grip_lost
- **Evidence:** Notebook 01 found all 54 rows with missing values also had a missing target, so after dropping those, the 14 raw eligible features have zero missing values. For them the imputer is a defensive step for production (a live sensor reading could arrive incomplete). The one column with real gaps is the engineered `total_joint_current_delta3` (720 NaNs, next entry), where the imputer does real work. `grip_lost` is a rare (~3.3%) binary flag; `RobustScaler`'s median/IQR both land on 0 for it, so scaling it would either silently no-op (via sklearn's internal IQR=0 guard) or be numerically meaningless. We exclude it from scaling via a `ColumnTransformer` passthrough branch instead of relying on that edge case.
- **Alternative considered:** Scale `grip_lost` along with everything else.
- **Why rejected:** Relying on an unstated library edge case (the IQR=0 fallback) instead of an explicit design choice is harder to defend, and a binary flag has no meaningful "scaled" interpretation.
- **Why median and RobustScaler:** the median and IQR are not moved by the large outlier values we deliberately keep (notebook 02, decision 1).

### Decision (reworded): Engineer total_joint_current and total_joint_current_delta3
- **Evidence:** `total_joint_current` is higher at stop rows than at normal rows (median 5.99 vs 4.31, notebook 03) and, unlike speed, is not a consequence of the stop. `total_joint_current_delta3` (change over the last 3 readings within the same cycle) captures whether the total current is rising or falling. Notebook 07 (section 2) shows it earns its place: removing it lowers the cross-validated PR-AUC of tuned XGBoost from 0.473 to 0.414, and the 16-feature model is ahead in 5 of 5 folds. It introduces 720 NaNs (3 per cycle × 240 cycles), the one place the median imputer does real work.
- **What it is not:** the original version of this entry justified the feature as capturing a "pre-stop rising-trend pattern". Notebook 03's same-cycle-position control shows that rise is mostly the normal start-of-cycle current wave, and notebook 07 (section 6b) shows the model uses delta3 mainly as a trend / cycle-phase signal (a fall pushes the prediction toward "stop"). It is not an early-warning precursor.
- **Alternative considered:** Use only the raw per-joint Current_J0-J5 columns without an aggregate or trend feature.
- **Why rejected:** the engineered features improve cross-validated PR-AUC (notebook 07), and the trend cannot be recovered from a single reading of the raw columns. The trend feature also means the deployed API needs an earlier reading (notebook 07, section 3).

### Decision (reworded): Keep all 16 features at this stage; deal with redundancy later
- **Evidence:** the correlation check above shows two very different situations. (1) The six temperature columns are near-duplicates of each other: all 15 pairs have |r| ≥ 0.99 (they move together with the time of day, notebook 07 section 4), and they are essentially uncorrelated with every other feature (|r| below 0.03). (2) Everything else is moderate: the strongest pairs are Current_J0/Current_J4 (−0.75), total_joint_current/total_joint_current_delta3 (0.72), total_joint_current/Current_J1 (−0.65) and Current_J1/Current_J2 (0.64). `total_joint_current` correlates negatively with Current_J1/J2 because it sums absolute values while the raw columns are signed and run consistently negative (medians about −2.2 and −1.1).
- **Alternative considered:** drop five of the six temperatures as redundant, and/or drop `total_joint_current` or `Current_J1`/`Current_J2`.
- **Why rejected at this stage:** correlation alone does not say which features help, tree models (XGBoost, Random Forest) tolerate correlated inputs, and the raw signed currents keep direction information the absolute-value aggregate loses. Selection was deferred to notebook 07, which tested feature groups directly and dropped the temperatures (and `grip_lost`) because they added nothing and behaved as a clock, leaving 9 features. The original version of this entry said "no pair exceeds |r| = 0.65"; that overlooked the temperature block and the delta3 pair.